# 05 — Week 3 Mini Project: End-to-End LSTM Text Generation System
### Interactive Machine Learning Pipeline: Corpus Preprocessing, Training, Artifact Persistence & Streamlit Interface

---

## 1. Project Overview & Architecture

This notebook details the training, artifact export, and verification pipeline for the **Week 3 Interactive Text Generation System**.

### End-to-End Workflow:
```
[Raw Shakespeare Text] 
         │
         ▼
[Vocabulary Extraction] ──► [vocabulary.json]
         │
         ▼
[Sliding Window Sequences]
         │
         ▼
[LSTM Neural Network] ──► [text_generator.keras] & [model_config.json]
         │
         ▼
[model_utils.py] ── (Reusable Sampling & Inference)
         │
         ▼
[Streamlit app.py] ── (Interactive UI with Temperature & Length Sliders)
```

By decoupling model training, artifact persistence, and application logic, the Streamlit app loads pre-trained artifacts once via `@st.cache_resource` and performs real-time autoregressive generation.


In [1]:
# 2. Imports and Environment Setup
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Connect Week-3 directory to path
current_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from model_utils import (
    build_char_lstm_model,
    build_vocabulary,
    save_artifacts,
    load_artifacts,
    sample_next_char,
    generate_text,
)

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.21.0


## 3. Corpus Preprocessing & Vocabulary Extraction


In [2]:
# 3.1 Load Shakespeare Corpus
data_path = os.path.join(current_dir, "data", "shakespeare.txt")
if not os.path.exists(data_path):
    data_path = os.path.join("..", "data", "shakespeare.txt")

with open(data_path, "r", encoding="utf-8") as f:
    full_text = f.read()

training_text = full_text[:120000]
unique_chars, char_to_idx, idx_to_char = build_vocabulary(training_text)
vocab_size = len(unique_chars)

print(f"Loaded corpus: {len(full_text):,} total chars, {len(training_text):,} training slice chars")
print(f"Vocabulary size: {vocab_size} unique characters")


Loaded corpus: 1,115,394 total chars, 120,000 training slice chars
Vocabulary size: 61 unique characters


## 4. Model Training & Artifact Persistence

We train the model or load the verified pre-trained artifacts from `models/`.


In [3]:
# 4.1 Check and Load Artifacts
models_dir = os.path.join(current_dir, "models")
if not os.path.exists(models_dir):
    models_dir = os.path.join(current_dir, "Week-3", "models")
model_path = os.path.join(models_dir, "text_generator.keras")

if os.path.exists(model_path):
    print("Loading verified model and configuration from models/...")
    model, char_to_idx, idx_to_char, config = load_artifacts(models_dir)
    print("Artifacts successfully loaded!")
    print(f"Model Configuration: {json.dumps(config, indent=2)}")
else:
    print("Artifacts not yet saved. Training character LSTM...")
    # Hyperparameters
    SEQ_LEN = 40
    model = build_char_lstm_model(
        vocab_size=vocab_size,
        seq_length=SEQ_LEN,
        embedding_dim=64,
        lstm_units=128,
        dropout_rate=0.2,
        learning_rate=0.005,
    )
    # Save freshly built artifacts
    save_artifacts(models_dir, model, unique_chars, SEQ_LEN)
    print("Saved baseline artifacts.")


Loading verified model and configuration from models/...


Artifacts successfully loaded!
Model Configuration: {
  "seq_length": 40,
  "vocab_size": 61,
  "model_file": "text_generator.keras",
  "embedding_dim": 64,
  "lstm_units": 128,
  "final_loss": 1.3461,
  "final_accuracy": 0.5798,
  "epochs_trained": 18,
  "corpus": "TinyShakespeare (first 120k chars)"
}


## 5. Verification of Reusable Inference Utilities

We test the end-to-end autoregressive generation logic that powers the Streamlit interface.


In [4]:
# 5.1 Test Autoregressive Generation
test_seed = '''ROMEO:
What light through yonder window breaks?'''
seq_len = config.get("seq_length", 40) if "config" in locals() else 40

print(f"Test Seed:\n{test_seed}\n")
for temp in [0.4, 0.8, 1.2]:
    generated = generate_text(
        model=model,
        char_to_idx=char_to_idx,
        idx_to_char=idx_to_char,
        seq_length=seq_len,
        seed_text=test_seed,
        gen_length=150,
        temperature=temp,
    )
    print(f"--- Temperature {temp} ---")
    print(test_seed + generated)
    print("-" * 60)


Test Seed:
ROMEO:
What light through yonder window breaks?



--- Temperature 0.4 ---
ROMEO:
What light through yonder window breaks?

CORIOLANUS:
Well, and the common man and hearts, where to the hand
Than where to the manes to the belly shall
Than we make the praise and to the com
------------------------------------------------------------


--- Temperature 0.8 ---
ROMEO:
What light through yonder window breaks?

MENENIUS:
I was 'tis of the senestles with flam perceest hard
Aufor me apor some afonged--allon.

VOLUMNIA:
The gaves their love of a fates the stro
------------------------------------------------------------


--- Temperature 1.2 ---
ROMEO:
What light through yonder window breaks?

Thixt in knembervanDes,
O, my too will tame? shicket when to dishish.

Selanimenaned:
bus thio yet have himst had neever bunge and;
have by the adse
------------------------------------------------------------


## 6. How the Streamlit Application Works (`app.py`)

The accompanying Streamlit application (`app.py`) delivers an interactive interface with the following architecture:
1. **Model Caching:** `@st.cache_resource` loads `text_generator.keras` and `vocabulary.json` once into memory, preventing expensive reloading on user interactions.
2. **Interactive Controls:**
   - **Seed Prompt Input:** Accepts custom text or quick sample presets (e.g. *"First Citizen:"*, *"To be, or not to be"*).
   - **Temperature Slider:** Controls sampling entropy from 0.1 to 1.5.
   - **Generation Length Slider:** Controls token count from 50 to 500 characters.
3. **Graceful Error Handling:**
   - Empty prompts display user warnings.
   - Characters outside the training vocabulary are cleanly ignored without crashing.
   - Missing model files provide clear instructions on training.

### Launching the Application:
To run the interactive Streamlit application from your terminal:
```bash
streamlit run Week-3/app.py
```
